# 🌿 AgriSense AI — Module 5: Part G (Augmentation & Evaluation of Leaf Classifier)
### Owner: G | Collaborator: F (Generator)
**Objective:** Evaluate quantitatively on the real test set whether synthetic leaf images produced by F (cVAE) improve MobileNetV2 classification performance on rare disease classes compared to baselines.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Clean 4-Way Experiment Grid:
| ID | Training Data Configuration | Purpose |
|---|---|---|
| **E0** | Real Only (Scarce Baseline) | "Before" benchmark |
| **E1** | Real + Classic Augmentation (Flip, Rotation, ColorJitter) | Control: Is generative better than standard cheap augmentation? |
| **E2** | Real + Class-Weighted Cross-Entropy Loss | Control: Handling class imbalance via loss weighting |
| **E3** | Real + Synthetic (from Part F's cVAE) | **Main Result:** Generative augmentation impact |

### Output Deliverables:
- `splits.csv` (70/15/15 stratified train/val/test split)
- `results/results.csv` (complete experimental comparison table)
- `results/per_class_f1.png` (clean before vs after bar chart on rare classes)
- `results/confusion_rare.png` (confusion matrix on rare classes)
- `results/notes.md` (conclusions & ExecuTorch export decision)


In [ ]:
# Step 1: Environment Setup & GPU Verification
import os
import sys
import json
import time
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Running on device: {device}")
if device.type == 'cuda':
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")


## 2. Unpack Part F Deliverables & Define Classes
We unpack `synthetic_dataset_for_G.zip` (delivered by Owner F) containing the 224x224 generated images for the 5 target classes.


In [ ]:
# Target Classes
TARGET_CLASSES = [
    "Potato___healthy",
    "Apple___Cedar_apple_rust",
    "Peach___healthy",
    "Raspberry___healthy",
    "Tomato___Tomato_mosaic_virus"
]
NUM_CLASSES = len(TARGET_CLASSES)
class_to_idx = {cls: idx for idx, cls in enumerate(TARGET_CLASSES)}
idx_to_class = {idx: cls for idx, cls in enumerate(TARGET_CLASSES)}

# Unpack Part F's zip deliverable if available
zip_path = "synthetic_dataset_for_G.zip"
synthetic_dir = Path("synthetic")

if os.path.exists(zip_path):
    print(f"[*] Extracting '{zip_path}'...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(".")
    print(f"[+] Unpacked synthetic dataset successfully.")
elif synthetic_dir.exists():
    print(f"[*] Using existing '{synthetic_dir}' directory.")
else:
    print("[!] 'synthetic' folder not found. Please upload 'synthetic_dataset_for_G.zip' or run train_cvae first.")

# Verify synthetic counts
for cls in TARGET_CLASSES:
    c_dir = synthetic_dir / cls
    c_count = len(list(c_dir.glob("*.png"))) if c_dir.exists() else 0
    print(f"    - Synthetic {cls}: {c_count} images")


## 3. Stratified Train / Val / Test Split (`splits.csv`)
- **Stratified 70% Train / 15% Val / 15% Test** using `random_state=42`.
- **Zero Leakage:** Asserts strictly zero overlap between splits.
- **Evaluation Rule:** Val and Test always contain **100% real images**.


In [ ]:
# Build or Load Real Dataset Split
def prepare_splits(data_root="plantvillage", splits_file="splits.csv"):
    if os.path.exists(splits_file):
        print(f"[*] Loading existing '{splits_file}'...")
        return pd.read_csv(splits_file)

    real_samples = []
    if os.path.exists(data_root):
        for cls_name in TARGET_CLASSES:
            c_dir = Path(data_root) / cls_name
            if c_dir.exists():
                for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
                    for img_p in c_dir.glob(ext):
                        real_samples.append({'path': str(img_p), 'class': cls_name})
    
    if not real_samples:
        print("[!] No PlantVillage folder found. Creating demo real set for pipeline verification...")
        demo_real = Path("demo_plantvillage")
        demo_real.mkdir(parents=True, exist_ok=True)
        colors = [(34, 139, 34), (139, 69, 19), (60, 179, 113), (46, 139, 87), (205, 92, 92)]
        for cls_name, col in zip(TARGET_CLASSES, colors):
            c_dir = demo_real / cls_name
            c_dir.mkdir(parents=True, exist_ok=True)
            for i in range(150):
                img = Image.new('RGB', (224, 224), color=(col[0] + random.randint(-20, 20), col[1] + random.randint(-20, 20), col[2] + random.randint(-20, 20)))
                p = c_dir / f"real_{i:04d}.png"
                img.save(p)
                real_samples.append({'path': str(p), 'class': cls_name})

    df = pd.DataFrame(real_samples)
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['class'], random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['class'], random_state=SEED)
    
    train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()
    train_df['split'] = 'train'; val_df['split'] = 'val'; test_df['split'] = 'test'
    
    splits_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
    splits_df.to_csv(splits_file, index=False)
    print(f"[+] Saved '{splits_file}' with {len(splits_df)} records.")
    return splits_df

splits_df = prepare_splits()

train_paths = set(splits_df[splits_df['split'] == 'train']['path'])
val_paths = set(splits_df[splits_df['split'] == 'val']['path'])
test_paths = set(splits_df[splits_df['split'] == 'test']['path'])

assert len(train_paths.intersection(val_paths)) == 0, "DATA LEAKAGE: train and val overlap!"
assert len(train_paths.intersection(test_paths)) == 0, "DATA LEAKAGE: train and test overlap!"
assert len(val_paths.intersection(test_paths)) == 0, "DATA LEAKAGE: val and test overlap!"
print("[+] Zero Data Leakage check PASSED successfully.")


## 4. Regime B: Simulating Extreme Disease Scarcity
We cap real training samples for target classes to **30 images** to simulate genuine class imbalance, making the evaluation sensitive to generative augmentation.


In [ ]:
SCARCE_COUNT = 30  # Cap real train images per class to 30

train_records = []
for cls_name in TARGET_CLASSES:
    cls_train = splits_df[(splits_df['split'] == 'train') & (splits_df['class'] == cls_name)]
    subsampled = cls_train.sample(n=min(SCARCE_COUNT, len(cls_train)), random_state=SEED)
    for _, row in subsampled.iterrows():
        train_records.append({'path': row['path'], 'class': row['class'], 'is_synth': 0})

real_train_df = pd.DataFrame(train_records)
val_df = splits_df[splits_df['split'] == 'val'].copy()
test_df = splits_df[splits_df['split'] == 'test'].copy()

print(f"[*] Regime B (Scarce) training samples: {len(real_train_df)} (strictly real)")
print(f"[*] Validation samples: {len(val_df)} (strictly real)")
print(f"[*] Test samples: {len(test_df)} (strictly real)")


## 5. Dataset Loader & Blur Shortcut Defense
In synthetic runs, we apply a subtle downscale/upscale blur to real images so MobileNetV2 cannot exploit softness artifacts (*"blurry = rare class"*).


In [ ]:
class MobileNetLeafDataset(Dataset):
    def __init__(self, records_df, transform=None, apply_blur_defense=False):
        self.records = records_df.reset_index(drop=True)
        self.transform = transform
        self.apply_blur_defense = apply_blur_defense

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img_path = row['path']
        cls_name = row['class']
        is_synth = row.get('is_synth', 0)
        label = class_to_idx[cls_name]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.apply_blur_defense and (not is_synth) and random.random() < 0.5:
            image = image.resize((64, 64), Image.BILINEAR).resize((224, 224), Image.BICUBIC)
            
        if self.transform:
            image = self.transform(image)
            
        return image, label

NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    NORMALIZE
])

classic_aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    NORMALIZE
])


## 6. MobileNetV2 Classifier Training Routine


In [ ]:
def get_mobilenet_model():
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
    return model.to(device)

def train_and_eval(model, train_loader, val_loader, test_loader, criterion, epochs=12, lr=1e-4):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_val_f1 = 0.0
    best_weights = None
    
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(labels.numpy())
                
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = model.state_dict()
            
    if best_weights:
        model.load_state_dict(best_weights)
        
    model.eval()
    test_preds, test_targets = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_targets.extend(labels.numpy())
            
    acc = accuracy_score(test_targets, test_preds)
    macro_f1 = f1_score(test_targets, test_preds, average='macro', zero_division=0)
    per_class = f1_score(test_targets, test_preds, average=None, zero_division=0)
    
    return acc, macro_f1, per_class, test_preds, test_targets


## 7. Execution of Experiments E0 to E3 (Clean 4-Way Comparison)
- **E0:** Real only (Baseline)
- **E1:** Real + Classic Augmentation
- **E2:** Real + Class-Weighted Loss
- **E3:** Real + Synthetic (cVAE) — **Main Result**


In [ ]:
# Prepare Test & Val Loaders (Always strictly real images)
val_ds = MobileNetLeafDataset(val_df, transform=base_transform)
test_ds = MobileNetLeafDataset(test_df, transform=base_transform)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Collect Synthetic Images from Part F
synth_records = []
for cls_name in TARGET_CLASSES:
    c_dir = synthetic_dir / cls_name
    if c_dir.exists():
        all_synth = list(c_dir.glob("*.png"))[:150]
        for p in all_synth:
            synth_records.append({'path': str(p), 'class': cls_name, 'is_synth': 1})
synth_df = pd.DataFrame(synth_records)
print(f"[*] Available synthetic images for training: {len(synth_df)}")

experiment_results = []
test_predictions_dict = {}
criterion_standard = nn.CrossEntropyLoss()

# ----------------- E0: Real Only Baseline -----------------
print("\n[*] Running E0: Real Only Baseline...")
ds_e0 = MobileNetLeafDataset(real_train_df, transform=base_transform)
acc_e0, f1_e0, pc_e0, preds_e0, targets_e0 = train_and_eval(
    get_mobilenet_model(), DataLoader(ds_e0, batch_size=32, shuffle=True), val_loader, test_loader, criterion_standard
)
experiment_results.append({'ID': 'E0', 'Config': 'Real Only (Baseline)', 'Accuracy': acc_e0, 'Macro_F1': f1_e0, 'Per_Class_F1': pc_e0})
test_predictions_dict['E0'] = preds_e0
print(f"    E0 Finished -> Macro-F1: {f1_e0:.4f} | Accuracy: {acc_e0:.4f}")

# ----------------- E1: Real + Classic Augmentation -----------------
print("\n[*] Running E1: Real + Classic Augmentation...")
ds_e1 = MobileNetLeafDataset(real_train_df, transform=classic_aug_transform)
acc_e1, f1_e1, pc_e1, preds_e1, targets_e1 = train_and_eval(
    get_mobilenet_model(), DataLoader(ds_e1, batch_size=32, shuffle=True), val_loader, test_loader, criterion_standard
)
experiment_results.append({'ID': 'E1', 'Config': 'Real + Classic Aug', 'Accuracy': acc_e1, 'Macro_F1': f1_e1, 'Per_Class_F1': pc_e1})
test_predictions_dict['E1'] = preds_e1
print(f"    E1 Finished -> Macro-F1: {f1_e1:.4f} | Accuracy: {acc_e1:.4f}")

# ----------------- E2: Real + Class Weighted Loss -----------------
print("\n[*] Running E2: Real + Class Weighted Loss...")
cls_counts = real_train_df['class'].value_counts()
weights = torch.tensor([1.0 / cls_counts[cls] for cls in TARGET_CLASSES], dtype=torch.float).to(device)
weights = weights / weights.sum()
criterion_weighted = nn.CrossEntropyLoss(weight=weights)

ds_e2 = MobileNetLeafDataset(real_train_df, transform=base_transform)
acc_e2, f1_e2, pc_e2, preds_e2, targets_e2 = train_and_eval(
    get_mobilenet_model(), DataLoader(ds_e2, batch_size=32, shuffle=True), val_loader, test_loader, criterion_weighted
)
experiment_results.append({'ID': 'E2', 'Config': 'Real + Class Weights', 'Accuracy': acc_e2, 'Macro_F1': f1_e2, 'Per_Class_F1': pc_e2})
test_predictions_dict['E2'] = preds_e2
print(f"    E2 Finished -> Macro-F1: {f1_e2:.4f} | Accuracy: {acc_e2:.4f}")

# ----------------- E3: Real + Synthetic -----------------
print("\n[*] Running E3: Real + Synthetic (cVAE Main Result)...")
train_e3_df = pd.concat([real_train_df, synth_df]).reset_index(drop=True)
ds_e3 = MobileNetLeafDataset(train_e3_df, transform=base_transform, apply_blur_defense=True)
acc_e3, f1_e3, pc_e3, preds_e3, targets_e3 = train_and_eval(
    get_mobilenet_model(), DataLoader(ds_e3, batch_size=32, shuffle=True), val_loader, test_loader, criterion_standard
)
experiment_results.append({'ID': 'E3', 'Config': 'Real + Synthetic (cVAE)', 'Accuracy': acc_e3, 'Macro_F1': f1_e3, 'Per_Class_F1': pc_e3})
test_predictions_dict['E3'] = preds_e3
print(f"    E3 Finished -> Macro-F1: {f1_e3:.4f} | Accuracy: {acc_e3:.4f}")


## 8. Export Clean Results Table & Charts
We compile:
1. `results/results.csv`: Table of Accuracy, Macro-F1, and per-class F1 across E0, E1, E2, E3.
2. `results/per_class_f1.png`: Clean 3-way bar chart: Baseline (E0) vs Classic Aug (E1) vs Synthetic (E3).
3. `results/confusion_rare.png`: Confusion matrix of E3 on real test data.
4. `results/notes.md`: Final conclusions and ExecuTorch recommendation.


In [ ]:
os.makedirs("results", exist_ok=True)

rows = []
for res in experiment_results:
    row = {
        'ID': res['ID'],
        'Configuration': res['Config'],
        'Accuracy': round(res['Accuracy'], 4),
        'Macro_F1': round(res['Macro_F1'], 4)
    }
    for idx, cls in idx_to_class.items():
        row[f'F1_{cls}'] = round(res['Per_Class_F1'][idx], 4)
    rows.append(row)

results_table_df = pd.DataFrame(rows)
results_table_df.to_csv("results/results.csv", index=False)
print("[+] Saved 'results/results.csv':")
display(results_table_df)

# Clean Bar Chart: E0 (Real Only) vs E1 (Classic Aug) vs E3 (Real + Synthetic)
plt.figure(figsize=(10, 5))
x = np.arange(NUM_CLASSES)
width = 0.28

plt.bar(x - width, results_table_df.iloc[0][[f'F1_{c}' for c in TARGET_CLASSES]], width=width, label='E0: Real Only (Baseline)', color='gray')
plt.bar(x, results_table_df.iloc[1][[f'F1_{c}' for c in TARGET_CLASSES]], width=width, label='E1: Classic Augmentation', color='steelblue')
plt.bar(x + width, results_table_df.iloc[3][[f'F1_{c}' for c in TARGET_CLASSES]], width=width, label='E3: Real + Synthetic (cVAE)', color='forestgreen')

plt.xlabel('Leaf Disease Class', fontweight='bold')
plt.ylabel('F1 Score on Real Test Set', fontweight='bold')
plt.title('Impact of Generative Augmentation on Rare Leaf Classes', fontweight='bold')
plt.xticks(x, [c.split('___')[1] if '___' in c else c for c in TARGET_CLASSES], rotation=15)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("results/per_class_f1.png", dpi=150)
plt.show()

# Confusion Matrix for E3
cm = confusion_matrix(targets_e0, test_predictions_dict['E3'])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=[c.split('___')[1] if '___' in c else c for c in TARGET_CLASSES],
            yticklabels=[c.split('___')[1] if '___' in c else c for c in TARGET_CLASSES])
plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.title('Real Test Confusion Matrix (E3: Real + Synthetic)', fontweight='bold')
plt.tight_layout()
plt.savefig("results/confusion_rare.png", dpi=150)
plt.show()

f1_diff = f1_e3 - f1_e0
re_export_decision = "YES - Re-export to ExecuTorch" if f1_diff > 0.02 else "NO - Keep existing on-device .pte (difference within noise)"

notes_content = f"""# AgriSense AI — Module 5 (Part G) Findings & Report

### 1. Key Results Summary:
- **Baseline (E0 - Scarce Real Only)**: Macro-F1 = {f1_e0:.4f}, Accuracy = {acc_e0:.4f}
- **Control 1 (E1 - Classic Aug)**: Macro-F1 = {f1_e1:.4f}, Accuracy = {acc_e1:.4f}
- **Control 2 (E2 - Class Weights)**: Macro-F1 = {f1_e2:.4f}, Accuracy = {acc_e2:.4f}
- **Main Result (E3 - Real + Synthetic)**: Macro-F1 = {f1_e3:.4f}, Accuracy = {acc_e3:.4f}

### 2. Relative Gain:
- Synthetic Augmentation delta over baseline: **+{f1_diff*100:.2f}% Macro-F1**

### 3. ExecuTorch Recommendation:
- **Decision:** {re_export_decision}
- **Reasoning:** In accordance with the ground rules, on-device models are only updated if the augmented model demonstrates clear improvement on rare classes without regression on common classes.
"""

with open("results/notes.md", "w") as f:
    f.write(notes_content)
print("[+] Saved 'results/notes.md'")

with zipfile.ZipFile("evaluation_results_G.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("results"):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, arcname=os.path.relpath(fp, "."))
    if os.path.exists("splits.csv"):
        zf.write("splits.csv", arcname="splits.csv")

print("[🎉] Part G Evaluation Complete! Deliverables saved in 'evaluation_results_G.zip'.")
try:
    from google.colab import files
    files.download("evaluation_results_G.zip")
except Exception:
    pass
